In [0]:

import requests
import json
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
base_path=f"abfss://sourcedata@abndemostorage.dfs.core.windows.net"
bronze_path=f"{base_path}/bronze/raw"
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled","true")

In [0]:

def fetch_shows():
    url = "https://api.tvmaze.com/shows"
    response = requests.get(url)
    response.raise_for_status()
    return response.json()
shows_data = fetch_shows()

def fetch_episodes(show_id):
    url = f"https://api.tvmaze.com/shows/{show_id}/episodes"
    response = requests.get(url)
    return response.json()

def fetch_cast(show_id):
    url = f"https://api.tvmaze.com/shows/{show_id}/cast"
    response = requests.get(url)
    return response.json()

episodes_data=[]
cast_data=[]
for show in shows_data[:50]:
    show_id = show["id"]
    try:
        episodes_data.extend(fetch_episodes(show_id))
        cast_data.extend(fetch_cast(show_id))
    except Exception as e:
        print(f"Error fetching episodes for show {show_id}: {e}")

In [0]:
import tempfile
import os

def write_json_lines(data, name):
    temp_dir = tempfile.gettempdir()
    path = os.path.join(temp_dir, f"{name}.jsonl")
    with open(path, 'w') as f:
        for item in data:
            f.write(json.dumps(item) + '\n')
    return f"file:{path}"

df_shows_data = spark.read.json(write_json_lines(shows_data, "shows"))
df_episodes_data = spark.read.json(write_json_lines(episodes_data, "episodes"))
df_cast_data = spark.read.json(write_json_lines(cast_data, "cast"))

In [0]:
#test
spark.conf.set(
  "fs.azure.account.key.abndemostorage.dfs.core.windows.net",
  "E03bAtGmc8RiJopddmUAmnuLk2+a0ITyIZu1I+V4wuscQrbPrTt3oUyHPXJf2XpcA4YL2vH1yqJP+AStde4D2w=="
)

In [0]:
#test
dbutils.fs.ls("abfss://sourcedata@abndemostorage.dfs.core.windows.net/")

In [0]:
df_shows_data.write.mode("overwrite").json(f"{bronze_path}/shows")
df_episodes_data.write.mode("overwrite").json(f"{bronze_path}/episodes")
df_cast_data.write.mode("overwrite").json(f"{bronze_path}/cast")


In [0]:
#test
display(df_shows_data.limit(10))
display(df_episodes_data.limit(10))
display(df_cast_data.limit(10))

In [0]:
df_shows = spark.read.json(f"{bronze_path}/shows")
df_shows.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"bronze_abn.shows")

df_episodes = spark.read.json(f"{bronze_path}/episodes")
df_episodes.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"bronze_abn.episodes")

df_cast = spark.read.json(f"{bronze_path}/cast")
df_cast.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"bronze_abn.cast")